# 📊 Stock Research Report Analyzer
### Longitudinal insights from your MOSL / AMBIT / AXIS / PL reports

**How to use:**
1. Run **Cell 1** once to install libraries (only needed first time)
2. Run **Cell 2** once to set your folder path and API key (only needed first time)
3. Run **Cell 3** every time — just change the ticker and number of quarters
4. Run **Cell 4** to process and get your HTML report

---

In [13]:
# ═══════════════════════════════════════════════════════
# CELL 1 — Install libraries (run once, then you can skip)
# ═══════════════════════════════════════════════════════
!pip install pypdf pdfplumber anthropic --quiet
print("✅ Libraries ready!")

✅ Libraries ready!


In [3]:
# ═══════════════════════════════════════════════════════
# CELL 2 — Your settings (edit once, then leave alone)
# ═══════════════════════════════════════════════════════

# Your folder where all FY26Q3, FY25Q4 etc. subfolders live
BASE_FOLDER = r"C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\ResearchReports\CompanyResearchReports"

# Your Anthropic API key — get free at https://console.anthropic.com
ANTHROPIC_API_KEY = "key"

# Model: haiku = cheapest (~₹0.10/report), sonnet = smarter (~₹0.50/report)
MODEL = "claude-haiku-4-5-20251001"

# Max pages to read per PDF (15–20 is plenty for most broker reports)
MAX_PAGES = 20

print(f"✅ Config set. Base folder: {BASE_FOLDER}")

✅ Config set. Base folder: C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\ResearchReports\CompanyResearchReports


In [15]:
# ═══════════════════════════════════════════════════════
# CELL 3 — ✏️  CHANGE THIS EVERY TIME YOU RUN
# ═══════════════════════════════════════════════════════

TICKER   = "ANUP"   # ← Change to your stock ticker
QUARTERS = 10           # ← How many past quarters to look back (8–12 recommended)
OUTPUT   = "html"       # ← "html" (recommended) or "text" or "both"

print(f"🎯 Will analyze: {TICKER}  |  Last {QUARTERS} quarters  |  Output: {OUTPUT}")

🎯 Will analyze: ANUP  |  Last 10 quarters  |  Output: html


In [17]:
# ═══════════════════════════════════════════════════════
# CELL 4 — Run the analysis (no edits needed here)
# ═══════════════════════════════════════════════════════
import os, re, json
from pathlib import Path
from datetime import datetime
import anthropic

# ── Helpers ──────────────────────────────────────────
BROKER_MAP = {
    "MOSL": "Motilal Oswal",
    "AMBIT": "Ambit Capital",
    "AXIS": "Axis Securities",
    "PL": "Prabhudas Liladhar",
    "KOTAK": "Kotak Securities",
}
QUARTER_RE = re.compile(r"^FY(\d{2})Q([1-4])$", re.IGNORECASE)
FILE_RE    = re.compile(r"^(FY\d{2}Q[1-4])_([A-Z0-9]+)_([A-Z]+)(?:_(\d+))?\.pdf$", re.IGNORECASE)

def quarter_key(name):
    m = QUARTER_RE.match(name)
    return (int(m.group(1)), int(m.group(2))) if m else (0, 0)

def discover_reports(base, ticker, max_q):
    base = Path(base)
    if not base.exists():
        raise FileNotFoundError(f"Folder not found: {base}")
    folders = sorted(
        [d for d in base.iterdir() if d.is_dir() and QUARTER_RE.match(d.name)],
        key=lambda d: quarter_key(d.name), reverse=True
    )
    reports, seen = [], 0
    for qf in folders:
        if seen >= max_q: break
        found = []
        for pdf in sorted(qf.glob("*.pdf")):
            m = FILE_RE.match(pdf.name)
            if m and m.group(2).upper() == ticker.upper():
                found.append({
                    "quarter": m.group(1).upper(), "ticker": m.group(2).upper(),
                    "broker": m.group(3).upper(),
                    "broker_name": BROKER_MAP.get(m.group(3).upper(), m.group(3).upper()),
                    "version": m.group(4) or "0",
                    "path": str(pdf), "filename": pdf.name
                })
        if found:
            reports.extend(found)
            seen += 1
            print(f"  ✅ {qf.name}: {len(found)} report(s)")
        else:
            print(f"  ⬜ {qf.name}: no {ticker} reports")
    return reports

def extract_text(pdf_path, max_pages):
    text = ""
    try:
        import pdfplumber
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages[:max_pages]:
                t = page.extract_text()
                if t: text += t + "\n\n"
        if text.strip(): return text
    except: pass
    try:
        from pypdf import PdfReader
        for page in PdfReader(pdf_path).pages[:max_pages]:
            t = page.extract_text()
            if t: text += t + "\n\n"
    except: pass
    return text

EXTRACT_PROMPT = """Analyze this broker research report and extract data as JSON only (no markdown).

Quarter: {quarter} | Ticker: {ticker} | Broker: {broker_name}

Report text:
{text}

Return ONLY this JSON (null for missing fields):
{{"rating":null,"target_price":null,"cmp":null,"target_price_upside_pct":null,
"revenue_actual":null,"revenue_estimate":null,"revenue_beat_miss":null,
"pat_actual":null,"pat_estimate":null,"pat_beat_miss":null,
"ebitda_margin_actual":null,"eps_actual":null,
"eps_fy_estimate":null,"eps_next_fy_estimate":null,
"management_guidance":null,
"key_positives":[],"key_negatives":[],
"analyst_view":null,"key_themes":[],
"previous_target_price":null,"rating_change":null}}"""

SYNTHESIS_PROMPT = """You are a senior equity analyst. Below is data from {n} research reports for {ticker} covering quarters: {quarters}.

Write a structured narrative insight with these sections:
1. **Financial Trajectory** — revenue, PAT, EPS trend; beats vs misses
2. **Analyst Confidence** — target price evolution, rating changes, what they signal
3. **Management Credibility** — guidance given vs outcomes in later quarters
4. **Recurring Themes** — what themes dominated early vs now
5. **Red & Green Flags** — consistent concerns or consistent strengths
6. **Current Consensus** — latest analyst view
7. **Honest Verdict** — 2-3 sentence plain-language take based on the track record

Write clearly for a retail investor who follows this stock closely.

Data:
{data}"""

def analyze_one(report, text, client):
    prompt = EXTRACT_PROMPT.format(
        quarter=report["quarter"], ticker=report["ticker"],
        broker_name=report["broker_name"], text=text[:12000]
    )
    try:
        msg = client.messages.create(model=MODEL, max_tokens=1200,
                                     messages=[{"role":"user","content":prompt}])
        raw = msg.content[0].text.strip()
        raw = re.sub(r"```json?\n?","",raw).replace("```","").strip()
        data = json.loads(raw)
        data["_ok"] = True
        return data
    except Exception as e:
        return {"_ok": False, "_error": str(e)}

def synthesize(ticker, reports, client):
    summaries = [{"quarter":r["quarter"],"broker":r["broker_name"],
                  "data":{k:v for k,v in r["analysis"].items() if not k.startswith("_")}}
                 for r in reports]
    quarters = ", ".join(sorted({r["quarter"] for r in reports}))
    prompt = SYNTHESIS_PROMPT.format(
        n=len(reports), ticker=ticker, quarters=quarters,
        data=json.dumps(summaries, indent=2)[:15000]
    )
    msg = client.messages.create(model=MODEL, max_tokens=3000,
                                 messages=[{"role":"user","content":prompt}])
    return msg.content[0].text

def build_html(ticker, reports, synthesis):
    rows = ""
    for r in reports:
        a = r["analysis"]
        rc = {"BUY":"#16a34a","ADD":"#65a30d","HOLD":"#d97706",
              "REDUCE":"#ea580c","SELL":"#dc2626"}.get(str(a.get("rating","")).upper(),"#6b7280")
        chg = a.get("rating_change","")
        chg_html = ""
        if chg:
            cc = {"UPGRADED":"#16a34a","DOWNGRADED":"#dc2626","MAINTAINED":"#6b7280"}.get(chg,"#6b7280")
            chg_html = f'<br><span style="color:{cc};font-size:0.7rem">▶ {chg}</span>'
        bm  = str(a.get("pat_beat_miss","")).lower()
        bmc = {"beat":"#dcfce7;color:#166534","miss":"#fee2e2;color:#991b1b",
               "in-line":"#fef9c3;color:#713f12"}.get(bm,"transparent;color:#64748b")
        tp = f"\u20b9{a.get('target_price','\u2014')}" if a.get('target_price') else '\u2014'
        cp = f"\u20b9{a.get('cmp','\u2014')}"         if a.get('cmp')          else '\u2014'
        up = f"{a.get('target_price_upside_pct','\u2014')}%" if a.get('target_price_upside_pct') else '\u2014'
        rows += f"""<tr>
          <td><b>{r['quarter']}</b></td><td>{r['broker_name']}</td>
          <td style="color:{rc};font-weight:700">{a.get('rating','\u2014')}{chg_html}</td>
          <td>{tp}</td><td>{cp}</td><td>{up}</td>
          <td>{a.get('pat_actual','\u2014')}</td>
          <td><span style="background:{bmc};padding:2px 7px;border-radius:999px;font-size:.75rem;font-weight:600">
              {a.get('pat_beat_miss','\u2014')}</span></td>
          <td>{a.get('eps_fy_estimate','\u2014')}</td>
          <td style="font-size:.8rem;color:#475569;max-width:220px">{str(a.get('management_guidance',''))[:150]}</td>
        </tr>\n"""

    syn_html = re.sub(r'\*\*(.*?)\*\*', r'<strong>\1</strong>', synthesis)
    syn_html = syn_html.replace("\n\n","</p><p>").replace("\n","<br>")

    return f"""<!DOCTYPE html><html lang='en'><head><meta charset='UTF-8'>
<title>{ticker} Research Insights</title>
<style>
  *{{box-sizing:border-box;margin:0;padding:0}}
  body{{font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;
        background:#f1f5f9;color:#1e293b;line-height:1.6}}
  .hdr{{background:linear-gradient(135deg,#1e3a5f,#2563eb);color:#fff;
        padding:2rem;display:flex;justify-content:space-between;align-items:center}}
  .hdr h1{{font-size:1.75rem;font-weight:700}}
  .hdr p{{opacity:.8;font-size:.9rem;margin-top:.2rem}}
  .badge{{background:rgba(255,255,255,.2);padding:.4rem .9rem;border-radius:999px;font-size:.85rem}}
  .wrap{{max-width:1280px;margin:0 auto;padding:1.5rem}}
  .card{{background:#fff;border-radius:12px;padding:1.5rem;
         box-shadow:0 1px 4px rgba(0,0,0,.08);margin-bottom:1.5rem}}
  h2{{font-size:1.1rem;font-weight:600;color:#1e3a5f;margin-bottom:1rem;
      padding-bottom:.5rem;border-bottom:2px solid #e2e8f0}}
  .syn{{font-size:.95rem;line-height:1.85;color:#334155}}
  .syn p{{margin-bottom:.9rem}}
  .syn strong{{color:#1e3a5f}}
  table{{width:100%;border-collapse:collapse;font-size:.84rem}}
  th{{background:#f8fafc;padding:.55rem .75rem;text-align:left;
      font-weight:600;color:#475569;white-space:nowrap;
      border-bottom:2px solid #e2e8f0}}
  td{{padding:.55rem .75rem;border-bottom:1px solid #e2e8f0;vertical-align:top}}
  tr:hover td{{background:#f8fafc}}
  .footer{{text-align:center;color:#94a3b8;font-size:.78rem;margin-top:1rem;padding-bottom:2rem}}
</style></head><body>
<div class='hdr'>
  <div><h1>\U0001f4ca {ticker} — Research Insights</h1>
  <p>{len(reports)} reports · {datetime.now().strftime('%d %b %Y, %H:%M')}</p></div>
  <div class='badge'>Powered by Claude AI</div>
</div>
<div class='wrap'>
  <div class='card'><h2>\U0001f9e0 Longitudinal Analyst Synthesis</h2>
    <div class='syn'><p>{syn_html}</p></div></div>
  <div class='card'><h2>\U0001f4cb Quarter-by-Quarter Summary</h2>
    <div style='overflow-x:auto'><table>
      <thead><tr><th>Quarter</th><th>Broker</th><th>Rating</th>
        <th>Target</th><th>CMP</th><th>Upside</th>
        <th>PAT Actual</th><th>vs Estimate</th><th>FY EPS Est</th>
        <th>Mgmt Guidance</th></tr></thead>
      <tbody>{rows}</tbody></table></div></div>
  <div class='footer'>Generated by Stock Research Analyzer · Claude AI · {datetime.now().strftime('%Y')}</div>
</div></body></html>"""

# ── Main flow ──────────────────────────────────────────
print(f"\n{'='*55}")
print(f"  Analyzing: {TICKER}  |  Last {QUARTERS} quarters")
print(f"{'='*55}\n")

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# Step 1 — find reports
print("📂 Scanning for reports...")
reports = discover_reports(BASE_FOLDER, TICKER, QUARTERS)
if not reports:
    print(f"\n❌ No reports found for '{TICKER}'. Check folder path and file names.")
else:
    print(f"\n✅ Found {len(reports)} report files\n")

    # Step 2 — extract + analyze each report
    print("🤖 Extracting and analyzing each report...\n")
    for r in reports:
        print(f"  → {r['filename']}")
        text = extract_text(r["path"], MAX_PAGES)
        if not text.strip():
            print("     ⚠️  No text extracted (scanned PDF?)")
            r["analysis"] = {"_ok": False, "_error": "no text"}
            continue
        a = analyze_one(r, text, client)
        r["analysis"] = a
        if a.get("_ok"):
            print(f"     ✅ Rating: {a.get('rating','?')}  Target: ₹{a.get('target_price','?')}  PAT: {a.get('pat_beat_miss','?')}")
        else:
            print(f"     ❌ {a.get('_error','unknown error')}")

    # Step 3 — synthesis
    good = [r for r in reports if r["analysis"].get("_ok")]
    print(f"\n✅ {len(good)}/{len(reports)} reports analyzed successfully")
    print("\n🧠 Generating longitudinal synthesis...")
    synthesis = synthesize(TICKER, good, client) if good else "No data available for synthesis."
    print("✅ Synthesis done\n")

    # Step 4 — save output
    ts   = datetime.now().strftime("%Y%m%d_%H%M")
    base = Path(BASE_FOLDER)

    if OUTPUT in ("html", "both"):
        html_path = base / f"{TICKER}_insights_{ts}.html"
        html_path.write_text(build_html(TICKER, reports, synthesis), encoding="utf-8")
        print(f"💾 HTML saved → {html_path}")

    if OUTPUT in ("text", "both"):
        lines = [f"{'='*65}", f"  {TICKER} — Research Insights", f"  {datetime.now().strftime('%d %b %Y, %H:%M')}", f"{'='*65}", "",
                 "SYNTHESIS", "-"*65, synthesis, "",
                 "QUARTER SUMMARY", "-"*65,
                 f"{'Quarter':<12}{'Broker':<22}{'Rating':<8}{'Target':>9}{'CMP':>9}{'PAT':>10}{'Beat/Miss':<12}"]
        for r in reports:
            a = r["analysis"]
            lines.append(f"{r['quarter']:<12}{r['broker_name']:<22}{str(a.get('rating','—')):<8}"
                         f"{'₹'+str(a.get('target_price','—')):>9}{'₹'+str(a.get('cmp','—')):>9}"
                         f"{str(a.get('pat_actual','—')):>10}{str(a.get('pat_beat_miss','—')):<12}")
        txt_path = base / f"{TICKER}_insights_{ts}.txt"
        txt_path.write_text("\n".join(lines), encoding="utf-8")
        print(f"💾 Text saved  → {txt_path}")

    print(f"\n🎉 Done! Open the file in your browser for the full report.")


  Analyzing: ANUP  |  Last 10 quarters

📂 Scanning for reports...
  ⬜ FY26Q3: no ANUP reports
  ⬜ FY26Q2: no ANUP reports
  ⬜ FY26Q1: no ANUP reports
  ⬜ FY25Q4: no ANUP reports
  ⬜ FY25Q3: no ANUP reports
  ⬜ FY25Q2: no ANUP reports
  ⬜ FY25Q1: no ANUP reports
  ⬜ FY24Q4: no ANUP reports
  ⬜ FY24Q3: no ANUP reports
  ⬜ FY24Q2: no ANUP reports
  ⬜ FY24Q1: no ANUP reports
  ⬜ FY23Q4: no ANUP reports
  ⬜ FY23Q3: no ANUP reports
  ⬜ FY23Q2: no ANUP reports
  ⬜ FY23Q1: no ANUP reports
  ⬜ FY22Q4: no ANUP reports
  ⬜ FY22Q3: no ANUP reports
  ⬜ FY22Q2: no ANUP reports
  ⬜ FY22Q1: no ANUP reports
  ⬜ FY21Q4: no ANUP reports
  ⬜ FY21Q3: no ANUP reports
  ⬜ FY21Q2: no ANUP reports
  ⬜ FY21Q1: no ANUP reports
  ⬜ FY20Q4: no ANUP reports
  ⬜ FY20Q3: no ANUP reports
  ⬜ FY20Q2: no ANUP reports
  ⬜ FY20Q1: no ANUP reports

❌ No reports found for 'ANUP'. Check folder path and file names.


---
## 💡 Tips

| Task | What to do |
|---|---|
| Different stock | Change `TICKER` in Cell 3 and re-run Cells 3 & 4 |
| More history | Increase `QUARTERS` (up to however many folders you have) |
| Smarter analysis | Change `MODEL` to `claude-sonnet-4-5-20250929` in Cell 2 (costs ~5x more) |
| Cheaper/faster | Reduce `MAX_PAGES` to 10 in Cell 2 |
| Output location | Reports save into your `BASE_FOLDER` automatically |

**File naming must follow:** `FY26Q3_TICKER_BROKER.pdf` (e.g. `FY26Q3_HDFCBANK_MOSL.pdf`)